In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.multioutput import RegressorChain
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score



In [2]:
df = pd.read_csv('preprocessed.csv')
df.set_index('Date', inplace=True)
cat = df.select_dtypes(include=['object']).columns.tolist()

x = df.drop(['Units Sold', 'Demand'], axis=1)
y = df[['Units Sold', 'Demand']]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

C:\Users\kumar\AppData\Local\Temp\ipykernel_32828\3298373648.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = df.select_dtypes(include=['object']).columns.tolist()


In [3]:
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), cat)
    ],
    remainder='passthrough'
)

x_train = ct.fit_transform(x_train)
x_test = ct.transform(x_test)

x_train[np.isinf(x_train)] = np.nan
x_test[np.isinf(x_test)] = np.nan

imputer = SimpleImputer(strategy='median')

x_train = imputer.fit_transform(x_train)
x_test = imputer.transform(x_test)


In [4]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.multioutput import RegressorChain
from xgboost import XGBRegressor

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 800, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 3),
        "gamma": trial.suggest_float("gamma", 0.0, 0.3, step=0.1),
        "subsample": trial.suggest_float("subsample", 0.8, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0, step=0.1),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 2.0, step=0.5),
        "random_state": 42,
        "n_jobs": -1,
    }

    xgb = XGBRegressor(**params)
    regressor_chain = RegressorChain(xgb, order=[0, 1])

    scores = cross_val_score(
        regressor_chain, x_train, y_train, cv=5, scoring="r2"
    )
    return scores.mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30) 



[I 2026-05-30 07:12:07,606] A new study created in memory with name: no-name-290097dd-b48c-4ed6-8d22-6cb5c1c83b0e
[I 2026-05-30 07:12:28,677] Trial 0 finished with value: 0.7890989702894826 and parameters: {'n_estimators': 600, 'learning_rate': 0.14310000289738825, 'max_depth': 5, 'min_child_weight': 2, 'gamma': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 2.0}. Best is trial 0 with value: 0.7890989702894826.
[I 2026-05-30 07:12:45,188] Trial 1 finished with value: 0.7055485659740247 and parameters: {'n_estimators': 700, 'learning_rate': 0.10913016089144635, 'max_depth': 3, 'min_child_weight': 3, 'gamma': 0.3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0}. Best is trial 0 with value: 0.7890989702894826.
[I 2026-05-30 07:13:02,773] Trial 2 finished with value: 0.7246716634322844 and parameters: {'n_estimators': 600, 'learning_rate': 0.08346590042851329, 'max_depth': 4, 'min_child_weight': 1, 'gamma': 0.2, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_l

In [5]:

print("\n--- Optimization Finished ---")
print(f"Best CV R2 Score: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


--- Optimization Finished ---
Best CV R2 Score: 0.8061
Best Hyperparameters:
  n_estimators: 750
  learning_rate: 0.14162465049355666
  max_depth: 5
  min_child_weight: 2
  gamma: 0.0
  subsample: 0.9
  colsample_bytree: 1.0
  reg_lambda: 1.5


In [6]:

best_xgb = XGBRegressor(**study.best_params, random_state=42)
final_chain = RegressorChain(best_xgb, order=[0,1])
final_chain.fit(x_train, y_train)

y_pred = final_chain.predict(x_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {np.sqrt(mse)}")
print(f"R2: {r2}")

RMSE: 19.281155676076093
R2: 0.8123667271004111


In [7]:
import joblib
joblib.dump(final_chain, 'xgb_model.pkl')

joblib.dump(ct,'ColumnTransformer.pkl')

joblib.dump(imputer,"imputer.pkl")

['imputer.pkl']